# 04: Feature engineering and ASI calculation

This notebook turns raw indicators into three stress signals and then combines them into the Affordability Stress Index (ASI). The ASI is a relative score: higher values mean more renter pressure compared with other metros in the same run.

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## Load processed data

Read the cleaned, merged dataset from `data/processed/`.

In [25]:
# Load the cleaned data
# df = pd.read_csv('../data/processed/metros_master.csv')

## Feature engineering

We create three signals that are easy to explain:
- Rent-to-income: how much of monthly income goes to rent.
- Rent growth: how fast rent is rising year over year.
- Vacancy stress: low vacancy means tight supply, so we flip the vacancy rate.

Each signal is scaled so metros can be compared on the same scale before we combine them.

In [26]:
from pathlib import Path

FEATURES_PATH = Path("data/processed/features_scaled.csv")
features_df = pd.read_csv(FEATURES_PATH)

feature_catalog = pd.DataFrame(
    [
        {
            "feature": "rent_to_income",
            "definition": "Avg 2BR rent ÷ monthly median after-tax income",
            "direction": "Higher = more stress",
            "scaled_column": "rent_to_income_scaled",
        },
        {
            "feature": "rent_growth_yoy",
            "definition": "Year-over-year % change in CMHC 2BR rent",
            "direction": "Higher = more stress",
            "scaled_column": "rent_growth_yoy_scaled",
        },
        {
            "feature": "vacancy_stress",
            "definition": "Negative CMHC vacancy rate (low vacancy ↦ stress)",
            "direction": "Higher = more stress",
            "scaled_column": "vacancy_stress_scaled",
        },
    ]
)

print(f"Loaded {len(features_df):,} metro-year observations from {FEATURES_PATH}")
feature_catalog

Loaded 41 metro-year observations from data/processed/features_scaled.csv


,feature,definition,direction,scaled_column
0,rent_to_income,Avg 2BR rent ÷ monthly median after-tax income,Higher = more stress,rent_to_income_scaled
1,rent_growth_yoy,Year-over-year % change in CMHC 2BR rent,Higher = more stress,rent_growth_yoy_scaled
2,vacancy_stress,Negative CMHC vacancy rate (low vacancy ↦ stress),Higher = more stress,vacancy_stress_scaled


In [27]:
raw_cols = ["rent_to_income", "rent_growth_yoy", "vacancy_rate", "vacancy_stress"]
scaled_cols = [f"{col}_scaled" for col in ["rent_to_income", "rent_growth_yoy", "vacancy_stress"]]

summary = (
    features_df[raw_cols + scaled_cols]
    .describe(percentiles=[0.25, 0.5, 0.75])
    .T
    .loc[:, ["mean", "std", "25%", "50%", "75%"]]
    .round(3)
)

summary

,mean,std,25%,50%,75%
rent_to_income,0.367,0.076,0.316,0.342,0.399
rent_growth_yoy,5.002,2.136,3.500,4.900,6.225
vacancy_rate,3.156,1.023,2.700,3.100,3.700
vacancy_stress,-3.156,1.023,-3.700,-3.100,-2.700
rent_to_income_scaled,0.300,0.921,-0.314,-0.000,0.686
rent_growth_yoy_scaled,0.037,0.784,-0.514,-0.000,0.486
vacancy_stress_scaled,-0.056,1.023,-0.600,0.000,0.400


## Calculate Affordability Stress Index (ASI)

We average the three scaled signals (with equal weights by default). The output is one score per metro. A higher score means more stress.

In [28]:
# Compute ASI scores using the shared script
!.venv/bin/python src/compute_asi.py \
    --input data/processed/features_scaled.csv \
    --output data/processed/asi_scores.csv \
    --plot-output data/processed/asi_top15.png


## Visualize index distribution

A quick plot shows the spread of ASI scores across metros. This is descriptive only; it does not explain the causes.

In [29]:
# Create visualizations of the ASI scores
# plt.figure(figsize=(10, 6))
# plt.hist(df['asi_score'], bins=30)
# plt.xlabel('ASI Score')
# plt.ylabel('Frequency')
# plt.title('Distribution of Affordability Stress Index')
# plt.savefig('../report/figures/asi_distribution.png', dpi=300, bbox_inches='tight')

## Save ASI scores

Write the scores to `data/processed/asi_scores.csv` for later notebooks and reports.

In [30]:
# Outputs written by the script:
# - ../data/processed/asi_scores.csv
# - ../data/processed/asi_top15.png
